# CMNIST demo viewer

This notebook loads a saved backend run from `examples/cmnist/results/default` and visualizes the main CMNIST diagnostics.

Run the backend first with:

```bash
python examples/cmnist/run.py --device cuda --output examples/cmnist/results/default
```

In [ ]:
from pathlib import Path
import sys, json
import torch
import matplotlib.pyplot as plt
import pandas as pd
from torchvision.utils import make_grid

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent.parent if (Path.cwd().parent.parent / "src").exists() else ROOT
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from deconfoundingfm.experimental import load_result_bundle

RESULT_DIR = ROOT / "examples" / "cmnist" / "results" / "default"
bundle = load_result_bundle(RESULT_DIR)
bundle.keys()

## Final metrics

In [ ]:
metrics = bundle["metrics"]
rows=[]
for method in ["source", "decfm", "ot"]:
    row={"method": method}
    for key in ["sw2", "fid"]:
        mkey=f"{method}_{key}"
        if mkey in metrics:
            row[key.upper()] = metrics[mkey]
    rows.append(row)
pd.DataFrame(rows)

## Sample grids

In [ ]:
samples = bundle["samples"]
def show_grid(x, title, nrow=8):
    grid = make_grid(x[:32].clamp(0,1), nrow=nrow).permute(1,2,0).cpu().numpy()
    plt.figure(figsize=(8,4))
    plt.imshow(grid)
    plt.title(title)
    plt.axis("off")

show_grid(samples["observed_a0"], "Observed P(Y|A=0)")
show_grid(samples["observed_a1"], "Observed P(Y|A=1)")
show_grid(samples["true_a0"], "True P(Y(0))")
show_grid(samples["true_a1"], "True P(Y(1))")
show_grid(samples["decfm_a0"], "DeconfoundingFM arm 0")
show_grid(samples["decfm_a1"], "DeconfoundingFM arm 1")
show_grid(samples["ot_a0"], "OT-DeconfoundingFM arm 0")
show_grid(samples["ot_a1"], "OT-DeconfoundingFM arm 1")
plt.show()

## SW2 convergence

In [ ]:
conv = bundle["convergence"]
plt.figure(figsize=(6,4))
plt.plot(conv["steps"], conv["decfm"], marker="o", label="DeconfoundingFM")
plt.plot(conv["steps"], conv["ot"], marker="o", label="OT-DeconfoundingFM")
plt.xlabel("updates")
plt.ylabel("SW2")
plt.legend()
plt.tight_layout()
plt.show()

## Color-distribution diagnostics

In [ ]:
color_diag = bundle.get("color_diagnostics", {})
if color_diag:
    rows=[]
    for method, vals in color_diag.items():
        row={"method": method}
        row.update(vals)
        rows.append(row)
    display(pd.DataFrame(rows))
else:
    print("No color diagnostics found.")

## Trajectory summary

In [ ]:
traj_summary = bundle.get("trajectory_summary", {})
if traj_summary:
    rows=[]
    for method, stats in traj_summary.items():
        avg = stats.get("average_over_arms", {})
        row={"method": method}
        row.update(avg)
        rows.append(row)
    display(pd.DataFrame(rows))
else:
    print("No trajectory summary found.")

## Example trajectories

In [ ]:
traj = bundle.get("trajectories")
if traj is None:
    print("No saved trajectories found.")
else:
    method = "decfm"
    arm = "arm1"
    seq = traj[method][arm]["trajectory"][:,0]  # first selected trajectory
    fig, axes = plt.subplots(1, min(len(seq),6), figsize=(14,3))
    idxs = torch.linspace(0, len(seq)-1, steps=min(len(seq),6)).long()
    for ax, idx in zip(axes, idxs):
        ax.imshow(seq[idx].permute(1,2,0).clamp(0,1).cpu().numpy())
        ax.set_title(f"t={idx.item()/(len(seq)-1):.2f}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()